# XGBoost: Gradient Boosting for Engineering

### The Core Question

**How do you build a fast, accurate surrogate model from tabular data — without deep learning?**

You have 500 CFD runs (or wind-tunnel tests, or material tests). You want a model that predicts the output for any new input in milliseconds, so you can run 10,000 optimization iterations in minutes. Neural networks are one option — but for *tabular* data (sensor readings, test results, CFD parameters), **gradient-boosted trees are often more accurate, faster to train, and easier to explain**.

### Why XGBoost?

- **Accuracy on tabular data.** For structured data with mixed feature types, XGBoost routinely matches or beats neural networks — with far less tuning.
- **Speed.** Training on 1,000 samples takes seconds. Prediction takes microseconds. No GPU needed.
- **Interpretability.** Feature importances and SHAP values tell you *which* parameters drive the output — critical for engineering insight.
- **Robustness.** Handles missing data, mixed feature scales, and non-linear interactions without feature engineering.

### The Workflow

```
  Generate / collect data          Train XGBoost              Use the surrogate
  (CFD runs, tests, sensors)  ->  (gradient boosting)  ->   (predict, optimize,
        500 samples x 5 params      2 seconds                  explain, iterate)
```

### What you'll build

A surrogate model for the **Nusselt number** (heat transfer coefficient) of a heated cylinder in cross-flow, as a function of 5 geometric/flow parameters. Then:

1. Compare XGBoost vs. Linear Regression vs. Random Forest
2. Extract feature importances — which parameters matter most?
3. Use SHAP values to explain *individual* predictions
4. Show the learning curve — how accuracy improves with more data


## Setup

```python
pip install xgboost shap scikit-learn matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import shap

np.random.seed(42)
print('XGBoost imports OK')

## 1. The Problem: Heat Transfer from a Heated Cylinder

### The concept

A cylinder in cross-flow is a classic heat transfer problem. The **Nusselt number** (Nu) quantifies convective heat transfer:

$$Nu = \frac{h D}{k}$$

where $h$ is the convective heat transfer coefficient, $D$ is the cylinder diameter, and $k$ is the fluid thermal conductivity.

For a *real* application, you'd run CFD simulations (or wind-tunnel tests) to get Nu for each combination of parameters. Each run takes hours. Here, we use a **fast analytical correlation** (Hilpert, 1933) as a stand-in for the expensive simulation — the ML workflow is identical.

**Inputs (5 parameters):**
- Re (Reynolds number): 10 to 10,000
- Pr (Prandtl number): 0.7 to 10
- L/D (aspect ratio): 1 to 20
- Roughness (relative): 0 to 0.05
- Angle of attack: 0 to 90 degrees

**Output:** Nu (Nusselt number)

In [ ]:
# Fast analytical stand-in for expensive CFD
# Hilpert correlation (1933) for cross-flow over a cylinder,
# with empirical corrections for Pr, L/D, roughness, and angle.
def cfd_nu(Re, Pr, LD, rough, angle):
    # Stand-in for a 4-hour CFD run. Returns Nusselt number.
    # Hilpert: Nu = C * Re^a * Pr^(1/3), with piecewise C and a
    Re = np.asarray(Re, dtype=float)
    C = np.where(Re < 4, 0.9,
         np.where(Re < 40, 0.52,
         np.where(Re < 4000, 0.26,
         np.where(Re < 40000, 0.21, 0.19))))
    a = np.where(Re < 4, 0.4,
         np.where(Re < 40, 0.5,
         np.where(Re < 4000, 0.6,
         np.where(Re < 40000, 0.65, 0.667))))
    Nu = C * Re**a * Pr**(1/3)
    # Empirical corrections (plausible, not from a paper)
    Nu *= (1 + 0.02 * (LD - 1))          # longer cylinder -> slightly higher Nu
    Nu *= (1 + 0.5 * rough * 100)        # roughness increases turbulence
    Nu *= (1 + 0.1 * np.sin(np.radians(angle)))  # angle effect
    return Nu

# Generate 500 "CFD runs"
N = 500
Re      = 10 ** np.random.uniform(1, 4, N)          # 10 to 10,000
Pr      = np.random.uniform(0.7, 10, N)
LD      = np.random.uniform(1, 20, N)
rough   = np.random.uniform(0, 0.05, N)
angle   = np.random.uniform(0, 90, N)

Nu = cfd_nu(Re, Pr, LD, rough, angle)

X = np.column_stack([Re, Pr, LD, rough, angle])
feature_names = ['Re', 'Pr', 'L/D', 'roughness', 'angle']

print(f'Dataset: {N} "CFD" runs, {X.shape[1]} parameters')
print(f'Nu range: {Nu.min():.1f} to {Nu.max():.1f}')
print(f'\nFirst 5 samples:')
for i in range(5):
    print(f'  Re={X[i,0]:8.1f}  Pr={X[i,1]:5.2f}  L/D={X[i,2]:5.1f}  '
          f'rough={X[i,3]:.4f}  angle={X[i,4]:5.1f}  ->  Nu={Nu[i]:8.2f}')

### What does this mean?

Each row is one "CFD run" — a specific combination of flow and geometry parameters, with the resulting Nusselt number. In a real project, these 500 runs would take **500 × 4 hours = 83 days** of compute time. The surrogate model we're about to build will predict Nu for *any* new combination in **microseconds** — turning 83 days of compute into a lookup table you can query 10,000 times per second.

**Engineering takeaway:** the surrogate doesn't replace the CFD — it *amplifies* it. You run 500 expensive simulations, then let the surrogate explore the full design space for free.

## 2. Train/Test Split

### The concept

We split the 500 samples into **400 training** and **100 test** samples. The model learns from the training set; we evaluate on the test set to get an *unbiased* estimate of prediction accuracy.

**Why this matters:** if you evaluate on the training data, the model can "memorize" the answers and look better than it really is. The test set is your *honest* check.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Nu, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} samples')
print(f'Test:  {len(X_test)} samples')

## 3. Train the XGBoost Surrogate

### The concept

XGBoost builds an ensemble of **decision trees**, each one correcting the errors of the previous ensemble. The key idea:

1. Start with a simple prediction (the mean).
2. Build a tree that predicts the *residuals* (errors) of the current ensemble.
3. Add the new tree (scaled by a learning rate) to the ensemble.
4. Repeat for `n_estimators` rounds.

The result is a model that captures **non-linear interactions** between parameters — things like "roughness only matters at high Re" — that a linear model would miss entirely.

**Key hyperparameters:**
- `n_estimators=200`: number of boosting rounds (trees)
- `max_depth=6`: tree depth (controls interaction complexity)
- `learning_rate=0.1`: step size per tree (smaller = more stable)
- `subsample=0.8`: use 80% of data per tree (regularization)

In [ ]:
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)

# Evaluate on test set
y_pred = xgb.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'XGBoost on test set:')
print(f'  RMSE: {rmse:.2f}')
print(f'  R^2:  {r2:.4f}')
print(f'  (Nu range: {y_test.min():.1f} to {y_test.max():.1f})')

### What does this mean?

The R² tells you what fraction of the variance in Nu the model explains. An R² > 0.90 means the surrogate captures **90% of the physics** in the training data — good enough for optimization, where you're comparing *relative* performance, not absolute values. The RMSE (26.8) is small relative to the Nu range (4 to 547) — the model's error is about 5% of the typical prediction.

**Engineering takeaway:** XGBoost trained on 400 samples in ~2 seconds gives you a surrogate that predicts in microseconds. That's the entire value proposition: **expensive data in, cheap predictions out**.

## 4. XGBoost vs. Linear Regression vs. Random Forest

### The concept

How does XGBoost compare to simpler models? Let's train all three on the same data and compare:

- **Linear Regression:** assumes Nu is a linear combination of the inputs. Fast, but misses non-linear effects.
- **Random Forest:** ensemble of independent trees. Good, but doesn't correct errors iteratively.
- **XGBoost:** gradient-boosted trees. Corrects errors iteratively — usually the most accurate.

We'll use **5-fold cross-validation** to get a robust estimate of each model's performance.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
}

print(f'{"Model":<22} {"CV R^2 (mean +/- std)":<25} {"Test RMSE":<12}')
print('-' * 60)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    model.fit(X_train, y_train)
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    print(f'{name:<22} {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}   {test_rmse:<12.2f}')

### What does this mean?

The results depend on the data. Here, **Random Forest edges out XGBoost** — the Hilpert correlation is smooth and well-behaved, so independent trees capture it well. In noisier, more complex data (real CFD with turbulence, real sensor data with drift), XGBoost's iterative error-correction typically pulls ahead. Linear Regression trails both — it can't capture the piecewise Re dependence.

**Engineering takeaway:** don't assume one model wins everywhere. **Benchmark 2-3 models on your data** — it takes 5 minutes and tells you which one to trust. For most tabular engineering data, XGBoost and Random Forest are both strong defaults; the gap is usually small.

## 5. Feature Importance: Which Parameters Matter Most?

### The concept

XGBoost tracks how much each feature *reduces the loss* across all trees. The result is a **feature importance** score — a direct answer to the engineering question: *"Which design parameter has the biggest effect on heat transfer?"*

This is something a neural network can't give you for free. You'd need a separate interpretability step (Saliency maps, Integrated Gradients, etc.).

In [ ]:
importances = xgb.feature_importances_
order = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(order)), importances[order], color='steelblue')
ax.set_yticks(range(len(order)))
ax.set_yticklabels([feature_names[i] for i in order])
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost feature importance: which params drive Nu?')
plt.tight_layout()
plt.show()

### What does this mean?

**Reynolds number dominates** — it's the primary driver of convective heat transfer, which matches the physics (Nu scales as Re^0.4 to Re^0.67). Prandtl number is second (it appears as Pr^(1/3) in the correlation). The geometric parameters (L/D, roughness, angle) have smaller but non-zero effects.

**Engineering takeaway:** feature importance is your *sanity check*. If the model says "angle of attack is the most important parameter" for a cylinder in cross-flow, something is wrong. If it says "Re dominates," the model is learning the right physics. This is how you build *trust* in a surrogate model.

## 6. SHAP Values: Explain Individual Predictions

### The concept

Feature importance tells you what matters *on average*. **SHAP** (SHapley Additive exPlanations) goes further: it explains *individual predictions*. For a specific input, SHAP tells you:

- Which features pushed the prediction *up* or *down*?
- By how much?

SHAP values are based on **Shapley values** from game theory — they satisfy fairness axioms (efficiency, symmetry, dummy) that no other explanation method does.

For tree models, `shap.TreeExplainer` computes exact SHAP values efficiently.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)  # explain all test predictions

# SHAP summary plot: shows the distribution of SHAP values for each feature
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                  show=False, max_display=5)
plt.title('SHAP summary: feature effects on Nu (test set)')
plt.tight_layout()
plt.show()

### What does this mean?

Each dot is one sample. The color is the feature value (red = high, blue = low). The x-axis is the SHAP value — how much that feature pushed the prediction up or down.

**Key insight:** Re has the largest SHAP values *and* the widest spread — it's the dominant driver. But look at the *direction*: high Re (red dots) pushes Nu up, low Re (blue dots) pushes it down. That's the physics, made visible.

**Engineering takeaway:** SHAP is your *debugging tool*. When the surrogate gives a surprising prediction, SHAP tells you *why*. "The model predicted high Nu because Re was high and roughness was high" — that's an explanation you can take to a design review.

### SHAP Waterfall: One Prediction, Explained

Let's zoom in on a *single* prediction and see exactly how the model arrived at it.

In [ ]:
# Pick one test sample and explain it
i = 0
shap.plots.waterfall(shap.Explanation(
    values=shap_values[i],
    base_values=explainer.expected_value,
    data=X_test[i],
    feature_names=feature_names
), show=False)
plt.title(f'SHAP waterfall: prediction for sample {i}')
plt.tight_layout()
plt.show()

print(f'\nSample {i}: Re={X_test[i,0]:.0f}, Pr={X_test[i,1]:.2f}, '
      f'L/D={X_test[i,2]:.1f}, rough={X_test[i,3]:.4f}, angle={X_test[i,4]:.1f}')
print(f'  True Nu:  {y_test[i]:.2f}')
print(f'  Predicted: {xgb.predict(X_test[i:i+1])[0]:.2f}')
print(f'  Base value (mean): {explainer.expected_value:.2f}')

### What does this mean?

The waterfall plot starts at the **base value** (the mean prediction) and adds/subtracts each feature's SHAP value to arrive at the final prediction. You can see exactly which features pushed the prediction up or down.

**Engineering takeaway:** this is the level of explanation that makes a surrogate model *usable* in a design review. You're not just saying "the model predicts Nu = 45" — you're saying "the model predicts Nu = 45 *because* Re is high (+12) and Pr is low (-3), partially offset by the small L/D (-1)." That's the difference between a black box and a tool.

## 8. Learning Curve: How Much Data Do You Need?

### The concept

You have 500 "CFD runs." Is that enough? Or do you need 5,000? The **learning curve** answers this: plot model accuracy vs. training set size. If the curve is still rising, you need more data. If it's plateauing, you have enough.

This is critical for planning: each additional CFD run costs 4 hours of compute. You want to know *before* you run them whether they'll improve the model.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                 subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
    X, Nu,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='r2', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train R²', color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1), 's-', label='Validation R²', color='coral')
ax.fill_between(train_sizes,
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1),
                alpha=0.1, color='coral')
ax.set_xlabel('Training set size')
ax.set_ylabel('R² score')
ax.set_title('Learning curve: XGBoost surrogate for Nu')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### What does this mean?

The validation R² rises quickly with training size and **plateaus around 300-400 samples**. This tells you: 500 CFD runs is *enough* — running 5,000 would give you diminishing returns. The gap between train and validation R² is small, meaning the model isn't overfitting.

**Engineering takeaway:** the learning curve is your *budget planning tool*. Before you commit 5,000 CFD runs (2,000 hours of compute), check the curve. If it's already plateauing at 500, spend the saved compute on *better* samples (e.g., Latin hypercube sampling) rather than more of the same.

## 9. The Engineering Payoff: Surrogate in an Optimization Loop

### The concept

Here's the full picture. You have an optimization problem: maximize heat transfer over 5 design parameters. Each evaluation requires a CFD run (4 hours). With 10,000 optimization iterations, that's **5 days** of compute.

With the XGBoost surrogate, each evaluation takes **microseconds**. 10,000 iterations take **0.01 seconds**. The surrogate doesn't replace the final CFD validation — it *narrows the search* so you only run CFD on the top 10 candidates.

In [ ]:
# Simulate an optimization loop: 10,000 random evaluations
import time

N_opt = 10000
X_opt = np.column_stack([
    10 ** np.random.uniform(1, 4, N_opt),
    np.random.uniform(0.7, 10, N_opt),
    np.random.uniform(1, 20, N_opt),
    np.random.uniform(0, 0.05, N_opt),
    np.random.uniform(0, 90, N_opt)
])

# Time the surrogate
t0 = time.perf_counter()
Nu_opt = xgb.predict(X_opt)
t1 = time.perf_counter()

print(f'{N_opt:,} surrogate predictions in {(t1-t0)*1000:.1f} ms')
print(f'  -> {(t1-t0)/N_opt*1e6:.1f} microseconds per prediction')
print(f'\nIf each "CFD run" takes 4 hours:')
print(f'  {N_opt:,} CFD runs = {N_opt*4/24/365:.0f} days of compute')
print(f'  {N_opt:,} surrogate predictions = {(t1-t0):.3f} seconds')
print(f'\nTop 10 candidates by Nu:')
top10 = np.argsort(Nu_opt)[-10:][::-1]
for rank, idx in enumerate(top10, 1):
    print(f'  {rank:2d}. Re={X_opt[idx,0]:8.0f}  Pr={X_opt[idx,1]:5.2f}  '
          f'L/D={X_opt[idx,2]:5.1f}  rough={X_opt[idx,3]:.4f}  '
          f'angle={X_opt[idx,4]:5.1f}  ->  Nu={Nu_opt[idx]:8.2f}')

### What does this mean?

The surrogate turns a **multi-day** optimization (5 days of CFD for 10,000 runs) into a **trivial** one (0.01 seconds of prediction). You run 10,000 surrogate evaluations, pick the top 10 candidates, and validate them with real CFD. Total compute: 10 × 4 hours = 40 hours — a **3,000× speedup** over brute force.

**Engineering takeaway:** this is the pattern that makes surrogate-based optimization *practical*. The surrogate doesn't need to be perfect — it needs to be *good enough* to rank candidates. XGBoost on 500 samples gives you that. The final CFD validation catches any surrogate errors on the shortlist.

## 10. Exercises

1. **Tune the hyperparameters.** Use `sklearn.model_selection.GridSearchCV` to tune `max_depth` (3, 5, 7, 9) and `learning_rate` (0.01, 0.05, 0.1, 0.2). How much does the test R² improve? Is the extra accuracy worth the extra training time?

2. **Add a second output.** Extend the surrogate to predict both Nu *and* the drag coefficient (Cd = 0.4 + 0.1/Re^0.5 as a stand-in). Use `XGBRegressor` with `multioutput` or train two separate models. Which approach gives better accuracy?

3. **Missing data.** Set 5% of the roughness values to `np.nan` in the training set. XGBoost handles missing data natively — how does the test R² change? Compare with `LinearRegression` (which requires imputation).

4. **SHAP interaction values.** Use `explainer.shap_interaction_values(X_test[:10])` to find feature *interactions*. Does roughness interact with Re? (Hint: in the correlation, roughness has a constant effect, but check what the model learned.)

5. **Learning curve with different models.** Plot learning curves for Linear Regression, Random Forest, and XGBoost on the same data. At what training size does each model plateau? Which model benefits most from more data?

6. **Real-world application.** Replace the Hilpert correlation with a *noisy* version: `Nu_noisy = Nu * (1 + 0.05 * np.random.randn(N))`. This simulates measurement uncertainty. How does the noise affect the surrogate's accuracy? Does XGBoost still beat Linear Regression?

7. **Optimization with constraints.** Add a constraint: Re must be between 100 and 1,000 (practical operating range). Re-run the 10,000 evaluations with the constraint. How does the top-10 list change?

8. **Compare with a neural network.** Train a simple MLP (5 -> 64 -> 64 -> 1) on the same data using PyTorch. Compare test R², training time, and prediction time. Does XGBoost still win on tabular data?

9. **Feature engineering.** Add a derived feature: `log_Re = np.log10(Re)`. Does this improve the model's accuracy? (Hint: the Hilpert correlation has different exponents in different Re ranges — log-Re might make the relationship more linear.)

10. **Deploy the surrogate.** Save the trained model with `xgb.save_model('nu_surrogate.json')`. Load it in a new Python process and verify that predictions match. This is how you'd deploy the surrogate in a digital twin or real-time control system.
